# 07. Arm 3: MRAC design (A0) and a PRELIMINARY run

**MRAC-RL logbook, step 3 of 3.** Arm 3 adds an adaptive inner loop that makes the *loaded* vehicle follow a **no-load reference model**. This notebook states the formulation (MQP §1.2, §4, §6), reports the feasibility gate for the chosen gains, and shows a **PRELIMINARY** run of the payload-aware law with the estimates ONLINE, FROZEN and FROZEN-after-learning, against PID+FF with mass-blind feedforward.

> **EXPLORATORY / offline / PROXY, and PRELIMINARY.** Arm 3 is design plus one short run per cell, three payload masses, one reference, gains chosen once by a stability scan (not by performance). Reduced taut point-load plant, development PC. **No claim of stability, convergence or benefit; no claim about the Pymunk chain plant.** Theory source: MQP (`.ai/research/derivations/mrac-quadrotor-payload.md`, status PROPOSED, not accepted). The double-integrator "Form 2" law of the formulation plan is **superseded** by the reviewed inertia-weighted B3 route (b) of MQP §4 and §6; it is not used here.

## 1. Research question

1. What exactly is the A0 architecture for arm 3, which uncertainty does it treat as matched and which not (MQP §1.3), and which derivation obligations are still open?
2. For the gains used here, is the pre-tuning feasibility gate $Q\succ0$ satisfied over a declared effectiveness family (MQP §6.7)?
3. In a short offline run with a payload the reference does not carry, what do the estimates do (ONLINE), and how does the loaded vehicle's tracking compare with PID+FF with a mass-blind feedforward and with the FROZEN twins?

## 2. Hypotheses and predicted direction

Stated before the run; the run is one deterministic episode per cell, so these are *directions*, not tests.

| # | Hypothesis | Predicted direction |
|---|---|---|
| H1 | The chosen gains pass the $Q$ gate over the declared family $\kappa,\kappa_\tau\in[0.9,1.1]$, and the original unit-test gains of `test_baselines.py` do not (the gate is a real constraint on $k_p,k_\theta,\lambda_\theta$) | margin $>0$ for the chosen gains; margin $<0$ for $k_p=40,k_\theta=5,\lambda_\theta=3$ |
| H2 | ONLINE payload estimates move away from their initial value 0 in the direction of the real payload mass (larger for 6 kg than for 2 kg) in the first seconds | $\hat m_L(t)>0$, ordered with $m_L$; **not** a claim of convergence to $m_L$ (the estimates are controller coefficients, MQP §6.3) |
| H3 | Under the payload mismatch ONLINE tracks the min-jerk reference at least as well as FROZEN (estimates fixed at their nominal values) | $c_{T,off}$(ONLINE) $\le c_{T,off}$(FROZEN) for the heavier masses |
| H4 | The law has no swing term, so it does **not** reduce swing relative to PID+FF | $c_{S,off}\gtrsim1$ for all arm-3 variants; this is why the softer law needs its own reference (section 4.5) |

## 3. Experiment identity

In [1]:
%matplotlib inline
import dataclasses, math, time
from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from mrac_lab import provenance, style, plots, widgets

prov = provenance.setup(pinned_commit="c16a4f8")
style.init()

from experiments.mrac.baselines import (
    canonical_offline_params, run_episode, MinJerkReference, PIDReferenceModel, EpisodeFailure,
    swing_equilibrium_off, S30_MASSES,
)
from experiments.mrac.offline_metrics import episode_metrics, c_T_off, c_S_off
from experiments.mrac.payload_law import (
    ClosedLoop, LoopState, PayloadLaw, PayloadLawParams, q_gate, q_gate_family, f_max_from_t_max,
)
from experiments.mrac.quad import QuadState

CANON = canonical_offline_params()
VEH, G = CANON.quad, CANON.quad.g
SOURCES = {r.name + " [" + r.unit + "]": "%s  <- %s" % (r.value, r.source) for r in CANON.records}
provenance.banner(prov, "07 Arm 3: MRAC design (A0) and a PRELIMINARY run", SOURCES,
                  notes=list(CANON.caveats) + ["PRELIMINARY: 3 payload masses, one reference, one run per cell; gains fixed once by a stability scan, not tuned for performance."])

07 Arm 3: MRAC design (A0) and a PRELIMINARY run
Label: EXPLORATORY / offline / PROXY (development PC, reduced taut point-load model)
Thesis checkout: C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load-mrac-w2c
Commit: c16a4f8986263d7323cf6299df1b10a5fce8b54e (branch mrac/w2c-baselines, clean for experiments/ and gym_pymunk_quadrotor/)
Python: 3.12.3
Claim limits: No stability claim, no Pymunk-transfer claim, no hardware claim. Offline metrics are named *_off and are not the canonical c_T / c_S.
import origin: experiments.mrac.quad -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load-mrac-w2c\experiments\mrac\quad.py
import origin: experiments.mrac.quad_load_pend -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load-mrac-w2c\experiments\mrac\quad_load_pend.py
import origin: experiments.notebook_u

m_q [kg],9.6 <- gym_pymunk_quadrotor/plant/config.py:61
width [m],1.164 <- gym_pymunk_quadrotor/plant/config.py:62
height [m],0.759 <- gym_pymunk_quadrotor/plant/config.py:63
J [kg m^2],1.5447816 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:94-95 (m (w^2+h^2)/12)
a_p = a_n [m],"0.582 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:96-97,301 (width/2)"
vehicle max_thrust (total) [N],"201.036 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:157-159 ((m+max_payload) g + m max_ascent_speed; config.py:66,69)"
T_max (per motor) [N],100.518 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:300; plant/entities/motor.py:135 (thrust = max_thrust_per_motor cmd^2)
tau_m [s],0.05 <- gym_pymunk_quadrotor/plant/config.py:72; motor.py:92-93 (first-order lag)
g [m/s^2],9.81 <- gym_pymunk_quadrotor/config/physics_legacy.py:25
kappa_L = kappa_R [-],1.0 <- no effectiveness split in Pymunk
d [m],"0.3795 <- gym_pymunk_quadrotor/plant/xpbd_cable.py:330-333 (pivot_local = (0, -height/2): cable leaves the bottom edge)"


## 4. Theory

### 4.1 Architecture A0 (MQP §1.2, PROPOSED)
Two parts: (1) a **frozen nominal reference branch**, here a *no-load* vehicle (`Quad`) driven by the package PID and a minimum-jerk reference (`PIDReferenceModel`); it produces a causal reference state, accelerations and requested wrench; (2) an **adaptive law on the uncertain plant** (`PayloadLaw` on `QuadLoadPend`) that reads the declared reference signals and *measured* plant signals $(p,v,\theta,\omega,\phi,\dot\phi)$, never the tension, accelerations or plant truth, and corrects the reference command so that the plant tracks the reference. It is **not** a full-state MRAC: the swing state is outside the storage function (MQP §1.5).

### 4.2 What is matched (MQP §1.3)
| Effect | Classification |
|---|---|
| vehicle mass, inertia, effectiveness, asymmetry | matched at the ideal interface $U=[F_y,F_z,\tau]$ (row-normalised, MQP §4.2) |
| cable moment $dT\sigma$; cable force along the thrust | matched at the ideal torque / thrust interface (MQP §5.8) |
| cable force across the thrust, $-T\sigma\,b'(\theta)$ | **not instantaneously matched**: only through attitude motion (tilt cascade) |
| swing dynamics, cable length $l$ | unmatched (swing); $l$ is a **mixed** perturbation, matched through $\mu_L=m_Ll$ |
| saturation, motor lag, sampling, cross-thrust | residuals, not parameters |

### 4.3 The law (MQP §6.2-6.5, §6.7, §6.9, §6.11)
Sliding variables $s_p=\dot e_p+\lambda_pe_p$, $s_\theta=\dot e_\theta+\lambda_\theta e_\theta$ (errors are plant minus reference). With $\Theta=[m_q,J,m_L,\mu_L]$ the route-(b) regressor $Y$ satisfies $Y\Theta=M\ddot\xi_{\rm ref}-\tfrac12\dot Ms+m_qg\bar e-m_L\bar T_0w$ (MQP §6.2), row-split into $Y_{p,p}$ ($2\times3$, parameters $[m_q,m_L,\mu_L]/\kappa$) and $Y_{p,\theta}$ ($1\times3$, $[J,m_L,\mu_L]/\kappa_\tau$). The cascade, in this order:
$$
\Delta F=Y_{p,p}\hat\Theta_p-k_ps_p-B_{r,p},\quad f_c=f_r+b(\theta_r)^{\top}\Delta F,\quad v=\frac{b'(\theta_r)^{\top}\Delta F}{f_r},\quad \vartheta=\theta_r+v,
$$
$$
\Delta\tau=Y_{p,\theta}\hat\Theta_\theta-k_\theta s_\theta-\hat\alpha f_c-B_{r,\theta},\qquad \tau_{cmd}=\tau_r+\Delta\tau,
$$
the tilt $v$ passing through a strictly proper filter (`filter_a`). Adaptation (continuous, before projection; sampled with $T_{adapt}=T_s$):
$$
\dot{\hat\Theta}_p=-\Gamma_pY_{p,p}^{\top}s_p,\qquad \dot{\hat\Theta}_\theta=-\Gamma_\theta Y_{p,\theta}^{\top}s_\theta,\qquad \dot{\hat\alpha}=\gamma_\alpha f_cs_\theta,
$$
each projected onto a closed box (payload lower bounds exactly 0). **ONLINE** = both flags `adapt_vehicle`, `adapt_payload` on; **FROZEN** = both off, with the nominal estimates ($m_q,J$ from the reference, **payload 0**); **FROZEN-learned** = the estimates at the end of an ONLINE run of the same condition, then held (the twin the W5 question needs). The vertical hover needs $(m_q+m_L)g$: the payload consumes thrust headroom (MQP §6.10).

**Reference with no payload (MQP §6.4, decision C-D4).** The reference carries no payload, so even with ideal estimates $\Delta F=0,\Delta\tau=0$ is *not* an identity for $m_L>0$: a positive-payload plant needs a nonzero correction. This is by design: the payload effect is a plant correction, and the law's "no-load identity" (MQP §6.12) is a law-level statement for $m_L=0$ only.

### 4.4 The $Q$ gate (MQP §6.7), a pre-tuning report
With $V_c=V_a+\tfrac c2e_\theta^2$ one gets $\dot V_c\le-y^{\top}Qy+y^{\top}r$ (continuous ideal model, projection and domain premises) with
$$
Q=\begin{bmatrix}\kappa k_p&0&-\kappa f_{\max}/2\\0&\kappa_\tau k_\theta&-c/2\\-\kappa f_{\max}/2&-c/2&c\lambda_\theta\end{bmatrix},\qquad
\text{margin}=\kappa_\tau k_\theta\lambda_\theta^{2}-\frac{\kappa f_{\max}^{2}}{4k_p}>0,
$$
and a shared $c\in(c_-,c_+)$ for a declared family; the worst member is $(\kappa_{\max},\kappa_{\tau,\min})$ (`q_gate_family`, $f_{\max}=2T_{\max}$). It is **reported, never enforced**, and proves nothing about the sampled, saturated closed loop (MQP §6.8: a conditional continuous-time scalar comparison only).

### 4.5 Why the softer law needs a nominal-payload reference, and what is still open
*Design reasoning (INFERRED, not verified here):* the softer law $G(-1,-0.5)$ adds a torque that moves the vehicle in response to the swing. Against a **no-load** reference (no swing state, no $\phi_{eq}$) the adaptive law reads that motion as tracking error and has an incentive to cancel it; a reference that already includes the softer law on a *nominal payload* would contain the intended damping, and the adaptive law would only correct the mismatch. This is why arm 3 in its final form is "softer law + MRAC" on a nominal-payload reference, and why the preliminary run below (no-load reference, no swing term) is only the first of the two steps.

**Open derivation obligations (MQP §6, §7; none done here):**
1. the reference bracket $B_r$ with payload columns for a nominal-payload reference (C-1 item 1, option b);
2. coercivity in the omitted swing state and a swing-bounded envelope for the $s$-only storage function;
3. projection containment, uniform reference/filter/residual bounds and forward completeness of the taut domain;
4. sampled-data (ZOH, $T_s=20$ ms) analysis: the continuous derivative calculation gives no descent inequality for finite steps (MQP §6.11, §4.7);
5. the frozen scales $S,P_0,T_0,\ell_*$ and the family bounds (MQP §6.8) before any comparison is presented;
6. the **reference-plant swing mismatch** $\phi-\phi_r$: a nominal-payload reference carries its own swing $\phi_r$, so it must be decided which swing the softer law acts on (the plant's or the reference's) and how the mismatch enters the residual partition (the $\epsilon_{inc}$ term, MQP §6.6).

## 5. Interactive exploration

### 5.1 The $Q$ gate for chosen gains (H1)
Sliders set $k_p,k_\theta,\lambda_\theta$ and the declared effectiveness family. The left panel shows the margin and the shared $c$ interval; the right panel the margin as a function of $k_\theta$ at the chosen $k_p$ and $\lambda_\theta$. $f_{\max}=2T_{\max}$ (conservative: both motors at their limit).

In [2]:
F_MAX = f_max_from_t_max(VEH.T_max)

def gate_numbers(k_p, k_theta, lam, k_lo, k_hi, kt_lo, kt_hi):
    fam = q_gate_family((k_lo, k_hi), (kt_lo, kt_hi), k_p, k_theta, lam, F_MAX)
    return fam

def draw_gate(k_p=20.0, k_theta=25.0, lam=5.0, kappa_lo=0.9, kappa_hi=1.1, kt_lo=0.9, kt_hi=1.1):
    fam = gate_numbers(k_p, k_theta, lam, kappa_lo, kappa_hi, kt_lo, kt_hi)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    kth = np.linspace(1.0, 60.0, 200)
    marg = [q_gate_family((kappa_lo, kappa_hi), (kt_lo, kt_hi), k_p, x, lam, F_MAX).worst_margin for x in kth]
    ax[0].plot(kth, marg, color="#0072B2"); ax[0].axhline(0, color=style.SLACK_COLOR, lw=0.9); ax[0].plot([k_theta], [fam.worst_margin], "o", color="#D55E00")
    ax[0].set_xlabel(r"$k_\theta$ [N m s/rad]"); ax[0].set_ylabel("worst-case margin"); ax[0].set_title(r"margin $=\kappa_\tau k_\theta\lambda_\theta^2-\kappa f_{max}^2/(4k_p)$", fontsize=8)
    cs = np.linspace(0.0, 2.2 * fam.c_star, 300)
    kap, kt = fam.worst_kappa, fam.worst_kappa_tau
    detQ = [np.linalg.det(np.array([[kap * k_p, 0, -kap * F_MAX / 2], [0, kt * k_theta, -c / 2], [-kap * F_MAX / 2, -c / 2, c * lam]])) for c in cs]
    ax[1].plot(cs, detQ, color="#0072B2"); ax[1].axhline(0, color=style.SLACK_COLOR, lw=0.9)
    if fam.c_interval:
        ax[1].axvspan(*fam.c_interval, color="#56B4E9", alpha=0.3, label="Q positive definite (leading minors), shared c"); ax[1].axvline(fam.c_star, **style.arm_style("ref", label="c*"))
    ax[1].set_xlabel("c [N m]"); ax[1].set_ylabel("det Q (worst member)"); ax[1].legend(fontsize=7); ax[1].set_title("determinant of Q against the weight c", fontsize=8)
    need = (VEH.m + 6.0) * G / kappa_lo
    ax[2].bar(["f_max = 2 T_max", "hover demand\n(m_q+6 kg) g / kappa_min"], [F_MAX, need], color=["#0072B2", "#D55E00"])
    ax[2].set_ylabel("N"); ax[2].set_title("f_max must also cover the hover demand (MQP 6.7)", fontsize=8)
    for a_ in ax: a_.grid(True, alpha=0.3)
    fig.suptitle("gate: %s; worst margin %.1f; shared c interval %s" % ("FEASIBLE" if fam.feasible else "NOT feasible", fam.worst_margin,
                 "(%.1f, %.1f)" % fam.c_interval if fam.c_interval else "empty"), fontsize=9)
    fig.tight_layout(); return fig

panel_gate = widgets.param_panel({
    "k_p": ("slider", 5.0, 200.0, 5.0, 20.0, "k_p [N s/m]"),
    "k_theta": ("slider", 1.0, 60.0, 1.0, 25.0, "k_theta [N m s/rad]"),
    "lam": ("slider", 1.0, 8.0, 0.5, 5.0, "lambda_theta [1/s]"),
    "kappa_lo": ("slider", 0.5, 1.0, 0.05, 0.9, "kappa min"),
    "kappa_hi": ("slider", 1.0, 1.5, 0.05, 1.1, "kappa max"),
    "kt_lo": ("slider", 0.5, 1.0, 0.05, 0.9, "kappa_tau min"),
    "kt_hi": ("slider", 1.0, 1.5, 0.05, 1.1, "kappa_tau max"),
}, draw_gate)

### 5.2 The run: setup
**Reference and plant.** The reference model is the **no-load** vehicle under the package PID tracking a rest-to-rest minimum-jerk transfer of 2 m in $y$ over 8 s from $(0,5.5)$, then holding (550 ticks = 11 s). The plant is the loaded `QuadLoadPend` at $L=2$ m, canonical $d=0.3795$ m, $m_L\in\{2,4,6\}$ kg, starting at the loaded hover equilibrium. Comparators: PID+FF with the true-mass feedforward (the denominator of the ratios) and PID+FF with the mass-blind feedforward (arm 1), both tracking the same minimum-jerk reference directly.

**Gains.** Fixed once, before comparing arms, by a 16-run stability scan at $m_L=4$ kg ($k_p\in\{20,35\}$, $k_\theta\in\{15,25\}$, $\lambda_\theta\in\{5,7\}$, filter $(s+a)^3$ with $a\in\{3,5\}$; the first set that stayed taut for the whole 11 s was kept). An earlier scan with $k_p\ge60$ left the taut domain within 2-5 s, so the gains below are on the soft side. They are **not tuned for performance**, and no other mass or reference was scanned.

In [3]:
LAW = dict(T_s=0.02, T_adapt=0.02, lambda_p=1.5, lambda_theta=5.0, k_p=20.0, k_theta=25.0,
           filter_a=(125.0, 75.0, 15.0),                       # (s + 5)^3
           d=CANON.payload.d, gamma_p=(0.05, 3.0, 3.0), gamma_theta=(1e-2, 1e-2, 1e-2), gamma_alpha=1e-4,
           p_ranges=((5.0, 15.0), (0.0, 10.0), (0.0, 25.0)),    # [m_q, m_L, mu_L]/kappa boxes
           theta_ranges=((0.5, 3.0), (0.0, 10.0), (0.0, 25.0)), # [J, m_L, mu_L]/kappa_tau boxes
           alpha_range=(-0.5, 0.5))
REF = MinJerkReference((0.0, 5.5), (2.0, 5.5), 8.0)
N_TICKS = 550
FAM = gate_numbers(LAW["k_p"], LAW["k_theta"], LAW["lambda_theta"], 0.9, 1.1, 0.9, 1.1)
OLD = gate_numbers(40.0, 5.0, 3.0, 0.9, 1.1, 0.9, 1.1)
display(pd.DataFrame([{"gains": k, **v} for k, v in {"chosen (k_p 20, k_theta 25, lambda_theta 5)": dict(feasible=FAM.feasible, worst_margin=FAM.worst_margin, c_minus=FAM.c_interval[0] if FAM.c_interval else None, c_plus=FAM.c_interval[1] if FAM.c_interval else None, c_star=FAM.c_star),
      "test_baselines.py gains (k_p 40, k_theta 5, lambda_theta 3)": dict(feasible=OLD.feasible, worst_margin=OLD.worst_margin, c_minus=None, c_plus=None, c_star=OLD.c_star)}.items()]).style.format(precision=2, na_rep="-").hide(axis="index"))
display(pd.DataFrame([{"parameter": k, "value": str(v)} for k, v in LAW.items()]).style.hide(axis="index"))

def make_loop(m_L, L=2.0):
    c = CANON.with_condition(m_L=m_L, cable_length=L)
    model = PIDReferenceModel(c.quad, c.allocator, c.gains, REF, c.timing)
    law = PayloadLaw(PayloadLawParams(**LAW), model)
    loop = ClosedLoop(law, c.plant(), c.allocator, c.timing)
    c0 = math.sqrt(c.quad.m * G / 2.0 / c.quad.T_max)              # no-load hover motor state of the reference vehicle
    start = LoopState(c.hover_state(position=(0.0, 5.5)), QuadState(0.0, 5.5, 0.0, 0.0, 0.0, 0.0, c0, c0), law.nominal_state())
    return c, loop, start

def _arrays(c, records, exit_, start_plant):
    n = len(records); dt = c.timing.T_s
    t = np.arange(n) * dt
    T_end = np.array([r.plant_step.end_tension for r in records])
    T0 = c.plant().accelerations(start_plant).tension
    T = np.concatenate([[T0], T_end[:-1]]) if n else np.array([])
    st = [r.next_state.plant for r in records]
    ref = [REF(ti) for ti in t]; prev = [REF(max(ti - dt, 0.0)) for ti in t]
    phi_eq = np.array([swing_equilibrium_off(a, b, dt, G)[0] for a, b in zip(ref, prev)])
    law = [r.next_state.law for r in records]
    lo_hi = LAW["p_ranges"] + LAW["theta_ranges"]
    est = np.array([list(s.theta_p) + list(s.theta_theta) + [s.alpha_hat] for s in law])
    hits = np.array([[abs(e - b[0]) < 1e-12 or abs(e - b[1]) < 1e-12 for e, b in zip(row[:6], lo_hi)] for row in est])
    return dict(
        t=t, p=np.array([[s.quad.p_y, s.quad.p_z] for s in st]), ref_p=np.array([[r.y, r.z] for r in ref]),
        theta=np.array([s.quad.theta for s in st]), phi=np.array([s.phi for s in st]), phi_dot=np.array([s.phi_dot for s in st]),
        phi_eq=phi_eq, T=T, T_end=T_end,
        f_req=np.array([r.command.f_req for r in records]), tau_req=np.array([r.command.tau_req for r in records]),
        f_alloc=np.array([sum(r.allocation.T_cmd) for r in records]), tau_alloc=np.array([c.allocator.a_n * (r.allocation.T_cmd[1] - r.allocation.T_cmd[0]) for r in records]),
        sat=np.array([any(r.allocation.saturated_low) or any(r.allocation.saturated_high) for r in records]),
        ref_model_p=np.array([[r.next_state.reference.p_y, r.next_state.reference.p_z] for r in records]),
        est=est, hits=hits, s_p=np.array([np.hypot(*r.command.s_p) for r in records]), s_theta=np.array([r.command.s_theta for r in records]),
        v=np.array([r.command.v for r in records]), eta0_minus_v=np.array([r.command.eta_0_minus_v for r in records]),
        f_c=np.array([r.command.f_c for r in records]), exit=exit_)

@lru_cache(maxsize=64)
def run_arm3(m_L, mode, L=2.0):
    # mode: online | frozen | frozen_learned
    c, loop, start = make_loop(m_L, L)
    if mode == "frozen_learned":
        learned = run_arm3(m_L, "online", L)["final_law"]
        start = dataclasses.replace(start, law=dataclasses.replace(start.law, theta_p=learned.theta_p, theta_theta=learned.theta_theta, alpha_hat=learned.alpha_hat))
    flag = mode == "online"
    records, exit_ = loop.run(start, N_TICKS, adapt_vehicle=flag, adapt_payload=flag)
    out = _arrays(c, records, exit_, start.plant) if records else {}
    out.update(final_law=records[-1].next_state.law if records else start.law, n=len(records), c=c, mode=mode, m_L=m_L)
    return out

@lru_cache(maxsize=32)
def run_pid(m_L, blind, L=2.0):
    c = CANON.with_condition(m_L=m_L, cable_length=L)
    ctrl = c.controller(m_hat=4.0 if blind else m_L)
    return run_episode(c.plant(), ctrl, REF, c.timing, N_TICKS, c.hover_state(position=(0.0, 5.5)))

def pid_record(r):
    return dict(t=r.t, p=r.p, ref_p=r.ref_p, theta=r.theta, phi=r.phi, phi_dot=r.phi_dot, phi_eq=r.phi_eq, T=r.T, f_req=r.f_req, tau_req=r.tau_req,
                f_alloc=r.f_alloc, tau_alloc=r.tau_alloc, sat=r.saturated)

def metrics_of(arr, m_L, template):
    # adapter: episode_metrics reads only t, p, ref_p, phi, phi_eq, T, T_end, failure, n_requested; replace exactly those on a template result
    ex = arr["exit"]
    fail = None if ex is None else EpisodeFailure(tick=ex.tick, time=ex.time, kind="taut_exit", source=ex.source, message="taut exit (tension %.3g N)" % ex.tension, substep=ex.substep, stage=ex.stage, stage_time=ex.time, tension=ex.tension)
    res = dataclasses.replace(template, t=arr["t"], p=arr["p"], ref_p=arr["ref_p"], phi=arr["phi"], phi_eq=arr["phi_eq"], T=arr["T"], T_end=arr["T_end"], failure=fail, n_requested=N_TICKS)
    return episode_metrics(res, m_L * G)

print("setup ok; cells below run ClosedLoop (about 6 s per 550-tick episode)")

gains,feasible,worst_margin,c_minus,c_plus,c_star
"chosen (k_p 20, k_theta 25, lambda_theta 5)",True,6.79,200.28,249.72,225.00
"test_baselines.py gains (k_p 40, k_theta 5, lambda_theta 3)",False,-237.36,-,-,27.00


parameter,value
T_s,0.02
T_adapt,0.02
lambda_p,1.5
lambda_theta,5.0
k_p,20.0
k_theta,25.0
filter_a,"(125.0, 75.0, 15.0)"
d,0.3795
gamma_p,"(0.05, 3.0, 3.0)"
gamma_theta,"(0.01, 0.01, 0.01)"


setup ok; cells below run ClosedLoop (about 6 s per 550-tick episode)


### 5.3 Arm-3 run explorer (PRELIMINARY)
Choose the payload mass and the variants. Panels: vehicle $y$ and $z$ against the commanded reference, swing $\phi$ with $\phi_{eq}$ (note the law has no swing term), tension against the slack line, requested against allocated wrench; below, the **adaptation telemetry**: the six row estimates ($[m_q,m_L,\mu_L]/\kappa$ and $[J,m_L,\mu_L]/\kappa_\tau$) and $\hat\alpha$ with the projection hits (ticks at a box bound), the sliding-variable norms $\|s_p\|,|s_\theta|$, the filtered tilt command $v$ and the filter error $\eta_0-v$. If a variant leaves the taut domain the episode ends there (a failure record, not hidden).

In [4]:
def draw_arm3(m_L=6.0, online=True, frozen=True, frozen_learned=False, pid_blind=True, pid_true=True):
    m_L = float(m_L); runs, notes = {}, []
    ref_rec = None
    if pid_true: runs["ff_true"] = pid_record(run_pid(m_L, False))
    if pid_blind: runs["pid_ff"] = pid_record(run_pid(m_L, True))
    for key, on, mode in (("mrac", online, "online"), ("frozen", frozen, "frozen"), ("frozen_learned", frozen_learned, "frozen_learned")):
        if on:
            a = run_arm3(m_L, mode)
            if a["exit"] is not None: notes.append("%s: taut exit at t = %.2f s" % (mode, a["exit"].tick * 0.02))
            if a["n"]: runs[key] = a
    style.LABEL_OVERRIDE.clear()
    first = next(iter(runs.values())); ref_rec = first if "ref_p" in first else None
    fig = plots.standard_figure(runs, T_nom=m_L * G, ref=ref_rec, title=f"arm 3 PRELIMINARY, m_L={m_L:g} kg, L=2 m, no-load reference")
    if notes: fig.text(0.5, 0.005, "; ".join(notes), ha="center", color=style.SLACK_COLOR, fontsize=8)
    return fig

panel_arm3 = widgets.param_panel({
    "m_L": ("dropdown", [2.0, 4.0, 6.0], 6.0, "payload m_L [kg]"),
    "online": ("check", True, "MRAC ONLINE"),
    "frozen": ("check", True, "MRAC FROZEN (nominal estimates)"),
    "frozen_learned": ("check", False, "MRAC FROZEN-learned"),
    "pid_blind": ("check", True, "arm 1 (PID+FF mass-blind)"),
    "pid_true": ("check", True, "true-mass PID+FF"),
}, draw_arm3)

In [5]:
def draw_telemetry(m_L=6.0, mode="online"):
    a = run_arm3(float(m_L), mode); t = a["t"]
    fig, ax = plt.subplots(2, 3, figsize=(13.5, 6.2))
    names = [r"$\hat m_q/\kappa$", r"$\hat m_L$ (force row)", r"$\hat\mu_L$ (force row)", r"$\hat J/\kappa_\tau$", r"$\hat m_L$ (torque row)", r"$\hat\mu_L$ (torque row)"]
    cols = ["#000000", "#D55E00", "#0072B2", "#999999", "#E69F00", "#56B4E9"]
    for i in (0, 1, 2): ax[0, 0].plot(t, a["est"][:, i], color=cols[i], label=names[i])
    ax[0, 0].axhline(m_L, **style.arm_style("ref", label="true $m_L$")); ax[0, 0].set_title("force-row estimates [kg, kg, kg m]", fontsize=8); ax[0, 0].legend(fontsize=6)
    for i in (3, 4, 5): ax[0, 1].plot(t, a["est"][:, i], color=cols[i], label=names[i])
    ax[0, 1].set_title("torque-row estimates [kg m^2, kg, kg m]", fontsize=8); ax[0, 1].legend(fontsize=6)
    ax[0, 2].plot(t, a["est"][:, 6], color="#CC79A7"); ax[0, 2].set_title(r"$\hat\alpha$ [m]", fontsize=8)
    for i in range(6):
        h = a["hits"][:, i]
        if h.any(): ax[1, 0].plot(t[h], np.full(h.sum(), i), "|", color=cols[i], ms=8)
    ax[1, 0].set_yticks(range(6), ["m_q", "m_L(p)", "mu(p)", "J", "m_L(th)", "mu(th)"], fontsize=7); ax[1, 0].set_title("projection hits (ticks at a box bound; the payload lower bound is 0)", fontsize=8)
    ax[1, 1].plot(t, a["s_p"], color="#0072B2", label=r"$\|s_p\|$ [m/s]"); ax[1, 1].plot(t, np.abs(a["s_theta"]), color="#D55E00", label=r"$|s_\theta|$ [rad/s]"); ax[1, 1].set_title("sliding variables", fontsize=8); ax[1, 1].legend(fontsize=7)
    ax[1, 2].plot(t, a["v"], color="#009E73", label="v (desired tilt increment)"); ax[1, 2].plot(t, a["eta0_minus_v"], color="#CC79A7", label=r"$\eta_0-v$ (filter error)"); ax[1, 2].set_title("tilt command and filter error [rad]", fontsize=8); ax[1, 2].legend(fontsize=7)
    for a_ in ax.ravel(): a_.set_xlabel("t [s]"); a_.grid(True, alpha=0.3)
    ex = a["exit"]; fig.suptitle("%s, m_L = %g kg; %s [PRELIMINARY; estimates are controller coefficients, not identified parameters]" % (mode, m_L, "taut exit at %.2f s" % (ex.tick * 0.02) if ex else "no exit in 11 s"), fontsize=9)
    fig.tight_layout(); return fig

panel_tel = widgets.param_panel({
    "m_L": ("dropdown", [2.0, 4.0, 6.0], 6.0, "payload m_L [kg]"),
    "mode": ("dropdown", ["online", "frozen_learned", "frozen"], "online", "variant"),
}, draw_telemetry)

### 5.4 Summary of the preliminary run
`c_T_off` and `c_S_off` are ratios to the **true-mass PID+FF** at the same mass (computed through `experiments.mrac.offline_metrics`, `_off`, not the canonical metrics). A taut exit leaves the ratio undefined (`failed = True`, shown as `nan`); the time of the exit is shown instead.

In [6]:
rows = []
for m_L in (2.0, 4.0, 6.0):
    base = run_pid(m_L, False); mt = episode_metrics(base, m_L * G)
    mb = episode_metrics(run_pid(m_L, True), m_L * G)
    variants = [("arm 1: PID+FF mass-blind", mb, None)]
    for mode, label in (("online", "MRAC ONLINE"), ("frozen_learned", "MRAC FROZEN-learned"), ("frozen", "MRAC FROZEN (nominal)")):
        a = run_arm3(m_L, mode); m = metrics_of(a, m_L, base) if a["n"] else None
        variants.append((label, m, a))
    for label, m, a in variants:
        ex = None if a is None or a["exit"] is None else a["exit"].tick * 0.02
        rows.append(dict(m_L=m_L, controller=label, c_T_off=c_T_off(m, mt) if m else np.nan, c_S_off=c_S_off(m, mt) if m else np.nan,
                         pos_rms_m=m.pos_rms_m if m else np.nan, swing_rms_deg=m.swing_rms_deg if m else np.nan, peak_T_over_Tnom=m.peak_T_over_Tnom if m else np.nan,
                         failed=bool(m.failed) if m else True, exit_time_s=ex if ex is not None else np.nan,
                         m_L_hat_force_row=a["final_law"].theta_p[1] if a is not None else np.nan, m_L_hat_torque_row=a["final_law"].theta_theta[1] if a is not None else np.nan))
SUMMARY = pd.DataFrame(rows)
display(Markdown("Ratios are relative to the **true-mass PID+FF** at the same condition (same reference, same 11 s). `pos_rms_m` is over the completed ticks only: a run that exits early is **not comparable** with a full run."))
display(SUMMARY.style.format({c: "{:.3f}" for c in SUMMARY.columns[2:7]} | {"m_L": "{:g}", "m_L_hat_force_row": "{:.2f}", "m_L_hat_torque_row": "{:.2f}", "exit_time_s": "{:.2f}"}, na_rep="n/a").hide(axis="index"))

Ratios are relative to the **true-mass PID+FF** at the same condition (same reference, same 11 s). `pos_rms_m` is over the completed ticks only: a run that exits early is **not comparable** with a full run.

m_L,controller,c_T_off,c_S_off,pos_rms_m,swing_rms_deg,peak_T_over_Tnom,failed,exit_time_s,m_L_hat_force_row,m_L_hat_torque_row
2,arm 1: PID+FF mass-blind,1.000,1.000,0.002,0.391,1.000,False,n/a,n/a,n/a
2,MRAC ONLINE,22.588,4.775,0.055,1.868,1.201,False,n/a,1.13,0.00
2,MRAC FROZEN-learned,106.231,3.474,0.259,1.359,1.013,False,n/a,1.13,0.00
2,MRAC FROZEN (nominal),249.931,6.224,0.608,2.434,1.031,False,n/a,0.00,0.00
4,arm 1: PID+FF mass-blind,1.000,1.000,0.003,0.358,1.000,False,n/a,n/a,n/a
4,MRAC ONLINE,30.689,9.845,0.105,3.526,1.382,False,n/a,8.13,0.00
4,MRAC FROZEN-learned,359.568,34.844,1.232,12.478,1.803,False,n/a,8.13,0.00
4,MRAC FROZEN (nominal),356.222,12.201,1.221,4.369,1.089,False,n/a,0.00,0.00
6,arm 1: PID+FF mass-blind,38.717,0.905,0.159,0.305,1.056,False,n/a,n/a,n/a
6,MRAC ONLINE,45.279,25.194,0.185,8.484,1.482,False,n/a,6.50,0.00


## 6. Observations
Computed from the executed cells.

In [7]:
obs = []
obs.append(f"**H1.** chosen gains: gate feasible = {FAM.feasible}, worst margin {FAM.worst_margin:.1f} N m (family kappa, kappa_tau in [0.9, 1.1], f_max = {F_MAX:.1f} N), shared c interval ({FAM.c_interval[0]:.1f}, {FAM.c_interval[1]:.1f}); "
           f"unit-test gains (40, 5, 3): feasible = {OLD.feasible}, margin {OLD.worst_margin:.1f}. The margin is small in relative terms: {FAM.worst_margin:.1f} of the first term {0.9 * LAW['k_theta'] * LAW['lambda_theta'] ** 2:.0f} (about {100 * FAM.worst_margin / (0.9 * LAW['k_theta'] * LAW['lambda_theta'] ** 2):.1f} %), so the gate is barely satisfied.")
est = {m: run_arm3(m, "online") for m in (2.0, 4.0, 6.0)}
obs.append("**H2.** ONLINE estimate of m_L (force row / torque row) at the end of the run, and the maximum over the run of the force-row estimate: " +
           "; ".join(f"{m:g} kg: {est[m]['est'][-1, 1]:.2f} / {est[m]['est'][-1, 4]:.2f}, max {est[m]['est'][:, 1].max():.2f}" for m in est) +
           ". The two rows are independent estimates and are not constrained to be equal or to converge to the true mass (MQP §6.3, §6.9).")
for m in (2.0, 4.0, 6.0):
    rr = SUMMARY[(SUMMARY.m_L == m)].set_index("controller")
    obs.append(f"**H3/H4, m_L = {m:g} kg.** c_T_off: arm 1 {rr.loc['arm 1: PID+FF mass-blind', 'c_T_off']:.2f}, ONLINE {rr.loc['MRAC ONLINE', 'c_T_off']:.2f}, FROZEN-learned {rr.loc['MRAC FROZEN-learned', 'c_T_off']:.2f}, FROZEN {rr.loc['MRAC FROZEN (nominal)', 'c_T_off']:.2f}; "
               f"c_S_off: arm 1 {rr.loc['arm 1: PID+FF mass-blind', 'c_S_off']:.2f}, ONLINE {rr.loc['MRAC ONLINE', 'c_S_off']:.2f}, FROZEN-learned {rr.loc['MRAC FROZEN-learned', 'c_S_off']:.2f}, FROZEN {rr.loc['MRAC FROZEN (nominal)', 'c_S_off']:.2f}; "
               f"failures (taut exit): ONLINE {bool(rr.loc['MRAC ONLINE', 'failed'])}, FROZEN-learned {bool(rr.loc['MRAC FROZEN-learned', 'failed'])}, FROZEN {bool(rr.loc['MRAC FROZEN (nominal)', 'failed'])}.")
osc = {m: (float(est[m]["est"][100:, 1].min()), float(est[m]["est"][100:, 1].max())) for m in est}
obs.append("**Estimate behaviour (ONLINE, force-row m_L, t > 2 s): range (min, max).** " + "; ".join(f"{m:g} kg: ({osc[m][0]:.2f}, {osc[m][1]:.2f})" for m in osc) +
           f". The box is [0, {LAW['p_ranges'][1][1]:g}]. An estimate that swings over most of its box at the swing frequency is driven by the swing (unmatched in the storage function, MQP §1.3), not by the payload mass: see the telemetry panel. The torque-row payload estimates stay at their lower bound 0 (final values in the table).")
hit_rows = {m: int(est[m]["hits"][:, 1:3].any(axis=1).sum()) for m in est}
obs.append("**Projection.** ticks with a force-row payload estimate at a box bound (ONLINE): " + ", ".join(f"{m:g} kg: {n}" for m, n in hit_rows.items()) + " (includes ticks at the lower bound 0 before the estimate leaves it).")
worse = {m: bool(SUMMARY[(SUMMARY.m_L == m) & (SUMMARY.controller == "MRAC ONLINE")].c_T_off.iloc[0] > SUMMARY[(SUMMARY.m_L == m) & (SUMMARY.controller == "arm 1: PID+FF mass-blind")].c_T_off.iloc[0]) for m in (2.0, 4.0, 6.0)}
obs.append("**ONLINE against arm 1.** c_T_off of ONLINE exceeds arm 1's at m_L = " + ", ".join(f"{m:g} kg: {v}" for m, v in worse.items()) + ". Arm 1 has a 4 kg payload feedforward, while FROZEN (nominal estimates) has no payload knowledge at all, so FROZEN is not a like-for-like frozen twin of arm 1; ONLINE is not shown to beat the baseline.")
n_exit = int(SUMMARY[SUMMARY.controller.str.startswith("MRAC")].failed.sum())
obs.append(f"**Failures.** {n_exit} of 9 arm-3 episodes ended in a taut exit within 11 s.")
display(Markdown("\n\n".join("- " + o for o in obs)))

- **H1.** chosen gains: gate feasible = True, worst margin 6.8 N m (family kappa, kappa_tau in [0.9, 1.1], f_max = 201.0 N), shared c interval (200.3, 249.7); unit-test gains (40, 5, 3): feasible = False, margin -237.4. The margin is small in relative terms: 6.8 of the first term 562 (about 1.2 %), so the gate is barely satisfied.

- **H2.** ONLINE estimate of m_L (force row / torque row) at the end of the run, and the maximum over the run of the force-row estimate: 2 kg: 1.13 / 0.00, max 4.14; 4 kg: 8.13 / 0.00, max 8.38; 6 kg: 6.50 / 0.00, max 10.00. The two rows are independent estimates and are not constrained to be equal or to converge to the true mass (MQP §6.3, §6.9).

- **H3/H4, m_L = 2 kg.** c_T_off: arm 1 1.00, ONLINE 22.59, FROZEN-learned 106.23, FROZEN 249.93; c_S_off: arm 1 1.00, ONLINE 4.78, FROZEN-learned 3.47, FROZEN 6.22; failures (taut exit): ONLINE False, FROZEN-learned False, FROZEN False.

- **H3/H4, m_L = 4 kg.** c_T_off: arm 1 1.00, ONLINE 30.69, FROZEN-learned 359.57, FROZEN 356.22; c_S_off: arm 1 1.00, ONLINE 9.84, FROZEN-learned 34.84, FROZEN 12.20; failures (taut exit): ONLINE False, FROZEN-learned False, FROZEN False.

- **H3/H4, m_L = 6 kg.** c_T_off: arm 1 38.72, ONLINE 45.28, FROZEN-learned 81.87, FROZEN 448.82; c_S_off: arm 1 0.91, ONLINE 25.19, FROZEN-learned 18.08, FROZEN 20.31; failures (taut exit): ONLINE False, FROZEN-learned False, FROZEN False.

- **Estimate behaviour (ONLINE, force-row m_L, t > 2 s): range (min, max).** 2 kg: (0.00, 4.14); 4 kg: (0.00, 8.38); 6 kg: (0.59, 10.00). The box is [0, 10]. An estimate that swings over most of its box at the swing frequency is driven by the swing (unmatched in the storage function, MQP §1.3), not by the payload mass: see the telemetry panel. The torque-row payload estimates stay at their lower bound 0 (final values in the table).

- **Projection.** ticks with a force-row payload estimate at a box bound (ONLINE): 2 kg: 147, 4 kg: 98, 6 kg: 59 (includes ticks at the lower bound 0 before the estimate leaves it).

- **ONLINE against arm 1.** c_T_off of ONLINE exceeds arm 1's at m_L = 2 kg: True, 4 kg: True, 6 kg: True. Arm 1 has a 4 kg payload feedforward, while FROZEN (nominal estimates) has no payload knowledge at all, so FROZEN is not a like-for-like frozen twin of arm 1; ONLINE is not shown to beat the baseline.

- **Failures.** 0 of 9 arm-3 episodes ended in a taut exit within 11 s.

## 7. Quantitative table

In [8]:
display(Markdown("Gate (section 5.2) and the arm-3 summary (section 5.4), repeated for the record. All ratios relative to the true-mass PID+FF at the same mass."))
display(pd.DataFrame([dict(item="Q gate, chosen gains", value="feasible" if FAM.feasible else "NOT feasible", margin=FAM.worst_margin, c_minus=FAM.c_interval[0], c_plus=FAM.c_interval[1]),
                      dict(item="Q gate, unit-test gains", value="feasible" if OLD.feasible else "NOT feasible", margin=OLD.worst_margin, c_minus=np.nan, c_plus=np.nan)]).style.format(precision=2, na_rep="-").hide(axis="index"))
display(SUMMARY.style.format({c: "{:.3f}" for c in SUMMARY.columns[2:7]} | {"m_L": "{:g}", "m_L_hat_force_row": "{:.2f}", "m_L_hat_torque_row": "{:.2f}", "exit_time_s": "{:.2f}"}, na_rep="n/a").hide(axis="index"))

Gate (section 5.2) and the arm-3 summary (section 5.4), repeated for the record. All ratios relative to the true-mass PID+FF at the same mass.

item,value,margin,c_minus,c_plus
"Q gate, chosen gains",feasible,6.79,200.28,249.72
"Q gate, unit-test gains",NOT feasible,-237.36,-,-


m_L,controller,c_T_off,c_S_off,pos_rms_m,swing_rms_deg,peak_T_over_Tnom,failed,exit_time_s,m_L_hat_force_row,m_L_hat_torque_row
2,arm 1: PID+FF mass-blind,1.000,1.000,0.002,0.391,1.000,False,n/a,n/a,n/a
2,MRAC ONLINE,22.588,4.775,0.055,1.868,1.201,False,n/a,1.13,0.00
2,MRAC FROZEN-learned,106.231,3.474,0.259,1.359,1.013,False,n/a,1.13,0.00
2,MRAC FROZEN (nominal),249.931,6.224,0.608,2.434,1.031,False,n/a,0.00,0.00
4,arm 1: PID+FF mass-blind,1.000,1.000,0.003,0.358,1.000,False,n/a,n/a,n/a
4,MRAC ONLINE,30.689,9.845,0.105,3.526,1.382,False,n/a,8.13,0.00
4,MRAC FROZEN-learned,359.568,34.844,1.232,12.478,1.803,False,n/a,8.13,0.00
4,MRAC FROZEN (nominal),356.222,12.201,1.221,4.369,1.089,False,n/a,0.00,0.00
6,arm 1: PID+FF mass-blind,38.717,0.905,0.159,0.305,1.056,False,n/a,n/a,n/a
6,MRAC ONLINE,45.279,25.194,0.185,8.484,1.482,False,n/a,6.50,0.00


## 8. Interpretation (EXPLORATORY, PRELIMINARY)

* **Formulation.** The A0 law corrects a no-load reference with payload columns in the regressor; the payload mass, the first moment $\mu_L=m_Ll$ and the vehicle coefficients are adaptive *coefficients* inside boxes, and the swing and the cable length stay unmatched. The $Q$ gate is a pre-tuning report that depends on $k_p,k_\theta,\lambda_\theta$ and the thrust bound $f_{\max}$: it **does** exclude some gain sets (the unit-test gains fail it), and it says nothing about the sampled, saturated loop.
* **The run.** What the table and plots show is listed in the observations; read them as one short deterministic episode per cell with soft gains chosen by a stability scan. In particular, (i) the estimates are controller coefficients and are not expected to equal the physical masses, (ii) the law has no swing term, so H4 is a statement about the design, (iii) a run that leaves the taut domain has no ratio and its position RMS covers fewer ticks. **No claim of benefit or stability follows.** Which of ONLINE, FROZEN-learned and FROZEN is better is a question for a protocol with several references, seeds-equivalents and a tuning rule fixed in advance (W5), not for this notebook.
* **Why the ratios are so large.** The denominator is the true-mass PID+FF following a slow 8 s minimum-jerk transfer almost perfectly (position RMS of a few millimetres, `pos_rms_m` column), so any other controller has a ratio of order $10$-$10^2$ in `c_T_off`, and a larger swing RMS in `c_S_off` (the law has no swing term). The ratios therefore say "much worse than an almost perfect tracker", not "unusable"; compare `pos_rms_m` in metres. The gains are soft by choice (the stability scan), and a large height error for FROZEN is consistent with the unmodelled weight acting against a soft $k_p$ (not separately checked).
* **Estimates.** The force-row payload estimate oscillates at the swing frequency over most of its box (section 6). This is **consistent with (INFERRED)** the estimate being driven by the swing rather than by the payload mass, and with ONLINE reducing the tracking error relative to FROZEN (nominal) through a swing-driven coefficient rather than by identifying $m_L$; FROZEN-learned (the end-of-run values held) being worse than FROZEN at 4 kg is consistent with the same reading. None of this was diagnosed: the excitation diagnostics of MQP §6.9 ($G_p$, $G_\theta$) were **not computed**, and the torque-row payload estimates stay pinned at their lower bound 0 with the cause **not examined**. H2 (ordering with $m_L$) is **not** supported by this run. The picture is consistent with, but not shown to be, the weak-excitation / unmatched-swing issue of MQP §6.9, and it is the motivation (INFERRED) for the nominal-payload reference with a swing term of section 4.5.
* **ONLINE is not better than the baseline here.** ONLINE has a larger `c_T_off` than arm 1 (mass-blind FF) at every mass in this run (section 6), FROZEN lacks the 4 kg payload knowledge that arm 1 has, and the gate margin is barely positive; no benefit over arms 1-2 is shown.
* **Link to arms 1-2.** Arm 3 as the plan states it (softer law plus MRAC) needs the nominal-payload reference of section 4.5 and the open derivation obligations listed there; the present run is the first step (MRAC with a no-load reference), compared with the mass-blind PID+FF of notebook 06.

## 9. Decisions and next steps

* **For Javier:** (i) whether the next step is the nominal-payload reference with the softer law (obligation 1 of section 4.5, a derivation task) before any further runs; (ii) whether $f_{\max}=2T_{\max}$ is the right bound for the $Q$ gate and which effectiveness family ($\kappa,\kappa_\tau\in[0.9,1.1]$ here is a placeholder) is declared; (iii) the independent review of the derivation of notebook 06 and of the formulation in section 4 is **not** done by these notebooks.
* **NOT RUN:** Pymunk episodes, training, wind or kick forcing, other references, gain sweeps for arm 3, a nominal-payload reference, any stability or convergence analysis of the sampled closed loop.